# Heart Disease Prediction â€” GeoXGB Pipeline

**Dataset**: `Heart_Disease_Prediction.csv` (270 samples, 13 features, binary target)

This notebook walks through four stages:

| Stage | Description |
|---|---|
| 1 | GeoXGB baseline â€” 5-fold CV (20 % holdout), parallel |
| 2 | HVRT augmentation â€” per-fold, generate 10 k synthetic samples, train GeoXGB, test on original holdout |
| 3 | Hyperparameter optimisation â€” grid search over the augmented pipeline, fully parallel |
| 4 | Final model â€” retrain on full dataset, predict `train.csv` (630 k rows), report AUC |

**Leakage prevention**: HVRT is fitted *only* on each fold's training partition; the holdout set is never seen during augmentation or training.

In [13]:
import warnings
warnings.filterwarnings("ignore")

import itertools
import multiprocessing
import time
from collections import defaultdict

import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder

import hvrt
from geoxgb import GeoXGBClassifier

# -- Global config -----------------------------------------------------------
N_FOLDS      = 5        # 20 % holdout per fold
RANDOM_STATE = 42
N_AUG        = 10_000   # synthetic samples to generate per fold
N_JOBS       = multiprocessing.cpu_count()

# GeoXGB recommended defaults (from benchmark study)
GEO_DEFAULTS = dict(
    n_rounds        = 1500,
    learning_rate   = 0.2,
    max_depth       = 3,
    refit_interval  = 20,
    min_samples_leaf= 5,
    reduce_ratio    = 0.7,
    auto_noise      = True,
    cache_geometry  = False,
    random_state    = RANDOM_STATE,
)

print(f"CPUs available : {N_JOBS}")
print(f"GeoXGB defaults: {GEO_DEFAULTS}")


CPUs available : 32
GeoXGB defaults: {'n_rounds': 1500, 'learning_rate': 0.2, 'max_depth': 3, 'refit_interval': 20, 'min_samples_leaf': 5, 'reduce_ratio': 0.7, 'auto_noise': True, 'cache_geometry': False, 'random_state': 42}


## Data Loading & Exploration

In [14]:
df = pd.read_csv("../data/Heart_Disease_Prediction.csv")

print(f"Shape  : {df.shape}")
print(f"Target : {df['Heart Disease'].value_counts().to_dict()}")
print()
display(df.describe().round(2))

Shape  : (270, 14)
Target : {'Absence': 150, 'Presence': 120}



,Age,Sex,Chest pain type,BP,Cholesterol,FBS over 120,EKG results,Max HR,Exercise angina,ST depression,Slope of ST,Number of vessels fluro,Thallium
count,270.00,270.00,270.00,270.00,270.00,270.00,270.00,270.00,270.00,270.00,270.00,270.00,270.00
mean,54.43,0.68,3.17,131.34,249.66,0.15,1.02,149.68,0.33,1.05,1.59,0.67,4.70
std,9.11,0.47,0.95,17.86,51.69,0.36,1.00,23.17,0.47,1.15,0.61,0.94,1.94
min,29.00,0.00,1.00,94.00,126.00,0.00,0.00,71.00,0.00,0.00,1.00,0.00,3.00
25%,48.00,0.00,3.00,120.00,213.00,0.00,0.00,133.00,0.00,0.00,1.00,0.00,3.00
50%,55.00,1.00,3.00,130.00,245.00,0.00,2.00,153.50,0.00,0.80,2.00,0.00,3.00
75%,61.00,1.00,4.00,140.00,280.00,0.00,2.00,166.00,1.00,1.60,2.00,1.00,7.00
max,77.00,1.00,4.00,200.00,564.00,1.00,2.00,202.00,1.00,6.20,3.00,3.00,7.00


In [15]:
# -- Preprocessing -----------------------------------------------------------
feature_cols = [c for c in df.columns if c != "Heart Disease"]
X = df[feature_cols].values.astype(np.float64)
le = LabelEncoder().fit(df["Heart Disease"])
y  = le.transform(df["Heart Disease"])       # Absence=0, Presence=1

print(f"Features : {X.shape[1]}  ({', '.join(feature_cols)})")
counts = {str(cls): int((y == i).sum()) for i, cls in enumerate(le.classes_)}
print(f"Classes  : {counts}")
print(f"Class balance: {100*y.mean():.1f} % Presence")

Features : 13  (Age, Sex, Chest pain type, BP, Cholesterol, FBS over 120, EKG results, Max HR, Exercise angina, ST depression, Slope of ST, Number of vessels fluro, Thallium)
Classes  : {'Absence': 150, 'Presence': 120}
Class balance: 44.4 % Presence


## Shared Utilities

In [16]:
def assign_labels_hvrt(h, X_syn, y_tr):
    """
    Partition-based label assignment for HVRT synthetic samples.

    Each synthetic sample is mapped to its HVRT partition (in z-score space
    via the fitted partition tree) and assigned the mean y of that partition
    from the real training data.  No k-NN in raw feature space needed.
    """
    X_syn_z      = h._to_z(X_syn)
    syn_part_ids = h.tree_.apply(X_syn_z)
    part_means   = {
        int(pid): float(y_tr[h.partition_ids_ == pid].mean())
        for pid in h.unique_partitions_
    }
    return np.array([round(part_means[int(pid)]) for pid in syn_part_ids], dtype=int)


def make_splits(X, y, n_folds=N_FOLDS, seed=RANDOM_STATE):
    """Return list of (train_idx, test_idx) from StratifiedKFold."""
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    return list(skf.split(X, y))


print("Utilities defined.")


Utilities defined.


---
## Stage 1 â€” GeoXGB Baseline

Train GeoXGB on each fold's original training partition (~216 samples).  
Evaluate on the 20 % holdout (~54 samples).  
All five folds run in parallel.

In [17]:
def _baseline_worker(fold_idx, X, y, splits, geo_params):
    """Evaluate GeoXGB baseline on one fold.  Runs in a subprocess."""
    import warnings as _w; _w.filterwarnings("ignore")
    import numpy as np
    from sklearn.metrics import roc_auc_score
    from geoxgb import GeoXGBClassifier

    tr, te = splits[fold_idx]
    m = GeoXGBClassifier(**geo_params)
    m.fit(X[tr], y[tr])
    auc = float(roc_auc_score(y[te], m.predict_proba(X[te])[:, 1]))
    return fold_idx, auc


splits = make_splits(X, y)

t0 = time.perf_counter()
baseline_results = Parallel(n_jobs=N_FOLDS, prefer="processes")(
    delayed(_baseline_worker)(fi, X, y, splits, GEO_DEFAULTS)
    for fi in range(N_FOLDS)
)
baseline_wall = time.perf_counter() - t0

baseline_aucs = [auc for _, auc in sorted(baseline_results)]

print(f"Wall time: {baseline_wall:.1f}s")
print()
df_base = pd.DataFrame(
    {"Fold": [f"Fold {i+1}" for i in range(N_FOLDS)], "AUC": [round(a, 5) for a in baseline_aucs]}
)
df_base.loc[N_FOLDS] = ["Mean / Std",
    f"{np.mean(baseline_aucs):.5f} / {np.std(baseline_aucs):.5f}"]
display(df_base.set_index("Fold"))

Wall time: 34.4s



,AUC
Fold,
Fold 1,0.82917
Fold 2,0.88889
Fold 3,0.82361
Fold 4,0.8875
Fold 5,0.90278
Mean / Std,0.86639 / 0.03314


---
## Stage 2 â€” HVRT Augmentation + GeoXGB

**Per-fold procedure (no leakage)**:
1. Fit HVRT on the training partition only.
2. Generate `N_AUG - n_train` synthetic samples via HVRT KDE (`expand`).
3. Assign class labels to synthetic samples via 5-NN majority vote.
4. Concatenate originals + synthetic -> 10 k training set.
5. Train GeoXGB on the 10 k set.
6. Evaluate on the original holdout fold (no synthetic data in test set).

HVRT is **never** fitted on test-fold samples.

In [18]:
def _hvrt_aug_worker(fold_idx, X, y, splits, geo_params, n_aug, seed):
    """
    HVRT-augmented fold evaluation.  Runs in a subprocess.
    Returns (fold_idx, auc, n_synthetic, hvrt_wall_s, geo_wall_s).
    """
    import warnings as _w; _w.filterwarnings("ignore")
    import time
    import numpy as np
    from sklearn.metrics import roc_auc_score
    import hvrt as _hvrt
    from geoxgb import GeoXGBClassifier

    tr, te = splits[fold_idx]
    X_tr, y_tr = X[tr], y[tr]

    # -- HVRT augmentation (partition-based label assignment) ----------------
    # min_samples_leaf=20: default auto-tune collapses to 1 partition for
    # n<~350 samples; 20 ensures >=n_features samples per partition for KDE.
    t_hvrt = time.perf_counter()
    n_syn = max(0, n_aug - len(X_tr))
    h = _hvrt.HVRT(min_samples_leaf=20, random_state=seed)
    h.fit(X_tr, y_tr.astype(float))

    if n_syn > 0:
        X_syn    = h.expand(n_syn)
        # Map each synthetic sample to its HVRT partition in z-space,
        # then assign the partition's mean y.  No k-NN in raw feature space.
        X_syn_z      = h._to_z(X_syn)
        syn_part_ids = h.tree_.apply(X_syn_z)
        part_means   = {
            int(pid): float(y_tr[h.partition_ids_ == pid].mean())
            for pid in h.unique_partitions_
        }
        y_syn    = np.array([round(part_means[int(p)]) for p in syn_part_ids], dtype=int)
        X_train  = np.vstack([X_tr, X_syn])
        y_train  = np.concatenate([y_tr, y_syn])
    else:
        X_train, y_train = X_tr, y_tr

    hvrt_wall = time.perf_counter() - t_hvrt

    # -- Train GeoXGB on augmented set ---------------------------------------
    # n_aug > min_train_samples=5000, so GeoXGB's auto_expand will not fire.
    t_geo = time.perf_counter()
    m = GeoXGBClassifier(**geo_params)
    m.fit(X_train, y_train)
    auc = float(roc_auc_score(y[te], m.predict_proba(X[te])[:, 1]))
    geo_wall = time.perf_counter() - t_geo

    return fold_idx, auc, n_syn, round(hvrt_wall, 2), round(geo_wall, 2)


t0 = time.perf_counter()
aug_results = Parallel(n_jobs=N_FOLDS, prefer="processes")(
    delayed(_hvrt_aug_worker)(fi, X, y, splits, GEO_DEFAULTS, N_AUG, RANDOM_STATE)
    for fi in range(N_FOLDS)
)
aug_wall = time.perf_counter() - t0

aug_results = sorted(aug_results, key=lambda r: r[0])
aug_aucs    = [r[1] for r in aug_results]

print(f"Wall time: {aug_wall:.1f}s")
print()
df_aug = pd.DataFrame([
    {"Fold": f"Fold {fi+1}", "AUC": round(r[1], 5),
     "Synthetic added": r[2], "HVRT (s)": r[3], "GeoXGB (s)": r[4]}
    for fi, r in enumerate(aug_results)
])
df_aug.loc[N_FOLDS] = [
    "Mean / Std",
    f"{np.mean(aug_aucs):.5f} / {np.std(aug_aucs):.5f}",
    aug_results[0][2], "", ""
]
display(df_aug.set_index("Fold"))


Wall time: 50.5s



,AUC,Synthetic added,HVRT (s),GeoXGB (s)
Fold,,,,
Fold 1,0.75139,9784,0.01,48.46
Fold 2,0.825,9784,0.01,50.47
Fold 3,0.76667,9784,0.01,50.45
Fold 4,0.87222,9784,0.01,49.96
Fold 5,0.82083,9784,0.01,48.65
Mean / Std,0.80722 / 0.04356,9784,,


In [19]:
# -- Baseline vs Augmented comparison ----------------------------------------
delta  = np.mean(aug_aucs) - np.mean(baseline_aucs)
winner = "HVRT-augmented" if delta > 0.001 else ("Baseline" if delta < -0.001 else "Tied")

df_cmp = pd.DataFrame({
    "Pipeline":   ["Baseline GeoXGB", "HVRT 10k + GeoXGB"],
    "Mean AUC":   [f"{np.mean(baseline_aucs):.5f}", f"{np.mean(aug_aucs):.5f}"],
    "Std AUC":    [f"{np.std(baseline_aucs):.5f}",  f"{np.std(aug_aucs):.5f}"],
    "Delta":      ["", f"{delta:+.5f}"],
})
print(f"Winner: {winner}")
print()
display(df_cmp.set_index("Pipeline"))

Winner: Baseline



,Mean AUC,Std AUC,Delta
Pipeline,,,
Baseline GeoXGB,0.86639,0.03314,
HVRT 10k + GeoXGB,0.80722,0.04356,-0.05917


---
## Stage 3 â€” Hyperparameter Optimisation

Grid search over GeoXGB parameters using the HVRT-augmented 5-fold CV pipeline.  
Every configuration is evaluated across all five folds in parallel.

**Search space** (informed by the GeoXGB benchmark study):

| Parameter | Values |
|---|---|
| `n_rounds` | 300, 1000 |
| `learning_rate` | 0.1, 0.2, 0.3 |
| `max_depth` | 3, 4 |
| `refit_interval` | 10, 20, 50 |

In [20]:
HPO_GRID = {
    "n_rounds":       [300, 1000],
    "learning_rate":  [0.1, 0.2, 0.3],
    "max_depth":      [3, 4],
    "refit_interval": [10, 20, 50],
}

# Fixed params not being swept
HPO_FIXED = {k: v for k, v in GEO_DEFAULTS.items() if k not in HPO_GRID}

hpo_configs = [
    dict(zip(HPO_GRID.keys(), vals))
    for vals in itertools.product(*HPO_GRID.values())
]

n_configs  = len(hpo_configs)
n_hpo_jobs = n_configs * N_FOLDS
print(f"Configs        : {n_configs}")
print(f"Folds per cfg  : {N_FOLDS}")
print(f"Total HPO jobs : {n_hpo_jobs}")
print(f"CPUs available : {N_JOBS}")

Configs        : 36
Folds per cfg  : 5
Total HPO jobs : 180
CPUs available : 32


In [21]:
def _hpo_worker(config_id, fold_idx, X, y, splits,
                hpo_params, fixed_params, n_aug, seed):
    """
    Evaluate one (config, fold) combination for HPO.
    Returns (config_id, fold_idx, auc).
    """
    import warnings as _w; _w.filterwarnings("ignore")
    import numpy as np
    from sklearn.metrics import roc_auc_score
    import hvrt as _hvrt
    from geoxgb import GeoXGBClassifier

    tr, te = splits[fold_idx]
    X_tr, y_tr = X[tr], y[tr]

    # HVRT augmentation (partition-based labels, fold-local, no leakage)
    n_syn = max(0, n_aug - len(X_tr))
    h = _hvrt.HVRT(min_samples_leaf=20, random_state=seed)
    h.fit(X_tr, y_tr.astype(float))

    if n_syn > 0:
        X_syn        = h.expand(n_syn)
        X_syn_z      = h._to_z(X_syn)
        syn_part_ids = h.tree_.apply(X_syn_z)
        part_means   = {
            int(pid): float(y_tr[h.partition_ids_ == pid].mean())
            for pid in h.unique_partitions_
        }
        y_syn    = np.array([round(part_means[int(p)]) for p in syn_part_ids], dtype=int)
        X_train  = np.vstack([X_tr, X_syn])
        y_train  = np.concatenate([y_tr, y_syn])
    else:
        X_train, y_train = X_tr, y_tr

    geo_params = {**fixed_params, **hpo_params}
    m = GeoXGBClassifier(**geo_params)
    m.fit(X_train, y_train)
    auc = float(roc_auc_score(y[te], m.predict_proba(X[te])[:, 1]))
    return config_id, fold_idx, auc


# All (config_id, fold_idx) jobs
hpo_jobs = [
    (cid, fi)
    for cid in range(n_configs)
    for fi  in range(N_FOLDS)
]

print(f"Launching {len(hpo_jobs)} HPO jobs across {N_JOBS} workers...")
t0 = time.perf_counter()

hpo_raw = Parallel(n_jobs=N_JOBS, prefer="processes", verbose=1)(
    delayed(_hpo_worker)(
        cid, fi, X, y, splits, hpo_configs[cid], HPO_FIXED, N_AUG, RANDOM_STATE
    )
    for cid, fi in hpo_jobs
)

hpo_wall = time.perf_counter() - t0
print(f"\nDone. Wall time: {hpo_wall:.1f}s")


Launching 180 HPO jobs across 32 workers...


[Parallel(n_jobs=32)]: Using backend LokyBackend with 32 concurrent workers.



Done. Wall time: 261.2s


[Parallel(n_jobs=32)]: Done 180 out of 180 | elapsed:  4.4min finished


In [22]:
# -- Aggregate HPO results ---------------------------------------------------
hpo_fold_aucs = defaultdict(list)
for cid, fi, auc in hpo_raw:
    hpo_fold_aucs[cid].append(auc)

hpo_rows = []
for cid, params in enumerate(hpo_configs):
    aucs = hpo_fold_aucs[cid]
    hpo_rows.append({
        **params,
        "mean_auc": round(float(np.mean(aucs)), 5),
        "std_auc":  round(float(np.std(aucs)),  5),
    })

df_hpo = pd.DataFrame(hpo_rows).sort_values("mean_auc", ascending=False)

print("Top 10 configurations:")
display(df_hpo.head(10).reset_index(drop=True))

# Best config
best_row    = df_hpo.iloc[0]
best_params = {
    k: int(best_row[k]) if k in ("n_rounds", "max_depth", "refit_interval")
    else float(best_row[k])
    for k in HPO_GRID
}
best_auc     = float(best_row["mean_auc"])
best_std     = float(best_row["std_auc"])

print(f"\nBest config   : {best_params}")
print(f"Best mean AUC : {best_auc:.5f} / std {best_std:.5f}")

Top 10 configurations:


,n_rounds,learning_rate,max_depth,refit_interval,mean_auc,std_auc
0,1000,0.3,3,50,0.81042,0.04530
1,1000,0.3,3,10,0.80972,0.04835
2,1000,0.2,3,20,0.80903,0.03718
3,1000,0.2,3,50,0.80792,0.03913
4,1000,0.3,3,20,0.80611,0.04559
5,1000,0.2,3,10,0.80528,0.03915
6,1000,0.3,4,50,0.80361,0.05022
7,1000,0.3,4,20,0.80361,0.04220
8,1000,0.2,4,20,0.80153,0.04765
9,1000,0.3,4,10,0.80111,0.04668



Best config   : {'n_rounds': 1000, 'learning_rate': 0.3, 'max_depth': 3, 'refit_interval': 50}
Best mean AUC : 0.81042 / std 0.04530


---
## Stage 4 â€” Final Model & Prediction on `train.csv`

Using the best HPO configuration:

1. Fit HVRT on the **full** 270-sample Heart Disease dataset.
2. Generate 10 k synthetic samples.
3. Train GeoXGB on those 10 k samples.
4. Load `train.csv` (630 k rows, same schema).
5. Predict and report AUC and AUC error.

In [23]:
# -- Final HVRT augmentation on full training set ----------------------------
print("Fitting HVRT on full 270-sample dataset...")
t0 = time.perf_counter()

h_final = hvrt.HVRT(min_samples_leaf=20, random_state=RANDOM_STATE)
h_final.fit(X, y.astype(float))

n_syn_final  = N_AUG - len(X)
X_syn_final  = h_final.expand(n_syn_final)

# Partition-based label assignment
X_syn_z_fin    = h_final._to_z(X_syn_final)
syn_parts_fin  = h_final.tree_.apply(X_syn_z_fin)
part_means_fin = {
    int(pid): float(y[h_final.partition_ids_ == pid].mean())
    for pid in h_final.unique_partitions_
}
y_syn_final = np.array([round(part_means_fin[int(p)]) for p in syn_parts_fin], dtype=int)

X_10k = np.vstack([X, X_syn_final])
y_10k = np.concatenate([y, y_syn_final])

hvrt_wall_final = time.perf_counter() - t0
print(f"  HVRT wall time  : {hvrt_wall_final:.2f}s")
print(f"  n_partitions    : {h_final.n_partitions_}")
print(f"  Training set    : {len(X_10k)} samples  "
      f"({len(X)} original + {n_syn_final} synthetic)")
print(f"  Class balance   : {100*y_10k.mean():.1f} % Presence")


Fitting HVRT on full 270-sample dataset...
  HVRT wall time  : 0.01s
  n_partitions    : 11
  Training set    : 10000 samples  (270 original + 9730 synthetic)
  Class balance   : 45.3 % Presence


In [24]:
# -- Train GeoXGB with best HPO config ---------------------------------------
final_params = {**HPO_FIXED, **best_params}
print(f"Training GeoXGB with best config: {best_params}")

t0 = time.perf_counter()
model_final = GeoXGBClassifier(**final_params)
model_final.fit(X_10k, y_10k)
geo_wall_final = time.perf_counter() - t0

print(f"  GeoXGB wall time : {geo_wall_final:.1f}s")
print(f"  Trees fitted     : {model_final.n_trees}")
print(f"  Resample events  : {model_final.n_resamples}")

Training GeoXGB with best config: {'n_rounds': 1000, 'learning_rate': 0.3, 'max_depth': 3, 'refit_interval': 50}
  GeoXGB wall time : 23.2s
  Trees fitted     : 1000
  Resample events  : 20


In [25]:
# -- Load train.csv and predict ----------------------------------------------
print("Loading train.csv (630 k rows)...")
t0 = time.perf_counter()
df_train = pd.read_csv("../data/train.csv")
load_wall = time.perf_counter() - t0
print(f"  Loaded {len(df_train):,} rows in {load_wall:.2f}s")

X_test_large = df_train[feature_cols].values.astype(np.float64)
y_test_large = le.transform(df_train["Heart Disease"])
print(f"  Class balance    : {100*y_test_large.mean():.1f} % Presence")

print("\nPredicting...")
t0 = time.perf_counter()
proba_large = model_final.predict_proba(X_test_large)[:, 1]
pred_wall   = time.perf_counter() - t0
print(f"  Prediction wall time: {pred_wall:.1f}s")

Loading train.csv (630 k rows)...
  Loaded 630,000 rows in 0.45s
  Class balance    : 44.8 % Presence

Predicting...
  Prediction wall time: 26.5s


In [26]:
# -- AUC report --------------------------------------------------------------
auc_final  = float(roc_auc_score(y_test_large, proba_large))
auc_error  = 1.0 - auc_final
cv_auc_aug = float(np.mean(aug_aucs))
gen_gap    = cv_auc_aug - auc_final

print("=" * 55)
print(f"  AUC  on train.csv (630 k)       : {auc_final:.5f}")
print(f"  AUC error  (1 - AUC)            : {auc_error:.5f}")
print("=" * 55)
print(f"\n  5-fold CV baseline              : {np.mean(baseline_aucs):.5f}")
print(f"  5-fold CV HVRT-augmented        : {cv_auc_aug:.5f}")
print(f"  5-fold CV best HPO config       : {best_auc:.5f}")
print(f"  Final (train.csv 630 k)         : {auc_final:.5f}")
print(f"  Generalisation gap (CV - large) : {gen_gap:+.5f}")

  AUC  on train.csv (630 k)       : 0.88789
  AUC error  (1 - AUC)            : 0.11211

  5-fold CV baseline              : 0.86639
  5-fold CV HVRT-augmented        : 0.80722
  5-fold CV best HPO config       : 0.81042
  Final (train.csv 630 k)         : 0.88789
  Generalisation gap (CV - large) : -0.08067


---
## Summary

In [27]:
# -- Final summary table -----------------------------------------------------
summary_rows = [
    {"Stage": "1 - Baseline GeoXGB (5-fold CV)",
     "Mean AUC": f"{np.mean(baseline_aucs):.5f}",
     "Std":      f"{np.std(baseline_aucs):.5f}",
     "Notes":    "Original 216-sample training fold"},
    {"Stage": "2 - HVRT 10k + GeoXGB (5-fold CV)",
     "Mean AUC": f"{np.mean(aug_aucs):.5f}",
     "Std":      f"{np.std(aug_aucs):.5f}",
     "Notes":    "216 real + 9784 synthetic per fold"},
    {"Stage": "3 - Best HPO config (5-fold CV)",
     "Mean AUC": f"{best_auc:.5f}",
     "Std":      f"{best_std:.5f}",
     "Notes":    str(best_params)},
    {"Stage": "4 - Final model -> train.csv (630k)",
     "Mean AUC": f"{auc_final:.5f}",
     "Std":      "N/A",
     "Notes":    f"AUC error = {auc_error:.5f}"},
]
display(pd.DataFrame(summary_rows).set_index("Stage"))

print("\nDesign notes:")
print("  - Leakage prevention: HVRT fitted per-fold on training data only")
print("  - Labels for synthetic samples: HVRT partition mean y (z-space tree)")
print("  - GeoXGB auto_expand inactive (10k > min_train_samples=5000)")
print("  - Parallelism: joblib Parallel(prefer='processes') for CV and HPO")


,Mean AUC,Std,Notes
Stage,,,
1 - Baseline GeoXGB (5-fold CV),0.86639,0.03314,Original 216-sample training fold
2 - HVRT 10k + GeoXGB (5-fold CV),0.80722,0.04356,216 real + 9784 synthetic per fold
3 - Best HPO config (5-fold CV),0.81042,0.04530,"{'n_rounds': 1000, 'learning_rate': 0.3, 'max_..."
4 - Final model -> train.csv (630k),0.88789,N/A,AUC error = 0.11211



Design notes:
  - Leakage prevention: HVRT fitted per-fold on training data only
  - Labels for synthetic samples: HVRT partition mean y (z-space tree)
  - GeoXGB auto_expand inactive (10k > min_train_samples=5000)
  - Parallelism: joblib Parallel(prefer='processes') for CV and HPO


---
## Stage 5 â€” XGBoost: Baseline vs HVRT Augmented

Same protocol as Stages 1â€“2 but with standard `XGBClassifier` instead of GeoXGB.  
This isolates whether the HVRT augmentation effect is model-specific.

**XGBoost settings** (matched to GeoXGB where applicable):
`n_estimators=1000, learning_rate=0.2, max_depth=4, subsample=0.8`

In [28]:
XGB_PARAMS = dict(
    n_estimators    = 1000,
    learning_rate   = 0.2,
    max_depth       = 4,
    subsample       = 0.8,
    colsample_bytree= 1.0,
    eval_metric     = "logloss",
    random_state    = RANDOM_STATE,
    verbosity       = 0,
)


def _xgb_baseline_worker(fold_idx, X, y, splits, xgb_params):
    """XGBoost baseline on one fold."""
    import warnings as _w; _w.filterwarnings("ignore")
    import numpy as np
    from sklearn.metrics import roc_auc_score
    from xgboost import XGBClassifier

    tr, te = splits[fold_idx]
    m = XGBClassifier(**xgb_params)
    m.fit(X[tr], y[tr])
    auc = float(roc_auc_score(y[te], m.predict_proba(X[te])[:, 1]))
    return fold_idx, auc


t0 = time.perf_counter()
xgb_base_results = Parallel(n_jobs=N_FOLDS, prefer="processes")(
    delayed(_xgb_baseline_worker)(fi, X, y, splits, XGB_PARAMS)
    for fi in range(N_FOLDS)
)
xgb_base_wall = time.perf_counter() - t0
xgb_base_aucs = [auc for _, auc in sorted(xgb_base_results)]

print(f"XGBoost baseline  --  wall time: {xgb_base_wall:.1f}s")
print()
df_xgb_base = pd.DataFrame({
    "Fold": [f"Fold {i+1}" for i in range(N_FOLDS)],
    "AUC":  [round(a, 5) for a in xgb_base_aucs],
})
df_xgb_base.loc[N_FOLDS] = [
    "Mean / Std",
    f"{np.mean(xgb_base_aucs):.5f} / {np.std(xgb_base_aucs):.5f}",
]
display(df_xgb_base.set_index("Fold"))

XGBoost baseline  --  wall time: 2.8s



,AUC
Fold,
Fold 1,0.87083
Fold 2,0.85972
Fold 3,0.81667
Fold 4,0.84028
Fold 5,0.91806
Mean / Std,0.86111 / 0.03391


In [29]:
def _xgb_aug_worker(fold_idx, X, y, splits, xgb_params, n_aug, seed):
    """HVRT 10k augmentation + XGBoost on one fold."""
    import warnings as _w; _w.filterwarnings("ignore")
    import numpy as np
    from sklearn.metrics import roc_auc_score
    import hvrt as _hvrt
    from xgboost import XGBClassifier

    tr, te = splits[fold_idx]
    X_tr, y_tr = X[tr], y[tr]

    n_syn = max(0, n_aug - len(X_tr))
    h = _hvrt.HVRT(min_samples_leaf=20, random_state=seed)
    h.fit(X_tr, y_tr.astype(float))

    if n_syn > 0:
        X_syn        = h.expand(n_syn)
        X_syn_z      = h._to_z(X_syn)
        syn_part_ids = h.tree_.apply(X_syn_z)
        part_means   = {
            int(pid): float(y_tr[h.partition_ids_ == pid].mean())
            for pid in h.unique_partitions_
        }
        y_syn    = np.array([round(part_means[int(p)]) for p in syn_part_ids], dtype=int)
        X_train  = np.vstack([X_tr, X_syn])
        y_train  = np.concatenate([y_tr, y_syn])
    else:
        X_train, y_train = X_tr, y_tr

    m = XGBClassifier(**xgb_params)
    m.fit(X_train, y_train)
    auc = float(roc_auc_score(y[te], m.predict_proba(X[te])[:, 1]))
    return fold_idx, auc


t0 = time.perf_counter()
xgb_aug_results = Parallel(n_jobs=N_FOLDS, prefer="processes")(
    delayed(_xgb_aug_worker)(fi, X, y, splits, XGB_PARAMS, N_AUG, RANDOM_STATE)
    for fi in range(N_FOLDS)
)
xgb_aug_wall = time.perf_counter() - t0
xgb_aug_aucs = [auc for _, auc in sorted(xgb_aug_results)]

print(f"XGBoost HVRT-augmented  --  wall time: {xgb_aug_wall:.1f}s")
print()
df_xgb_aug = pd.DataFrame({
    "Fold": [f"Fold {i+1}" for i in range(N_FOLDS)],
    "AUC":  [round(a, 5) for a in xgb_aug_aucs],
})
df_xgb_aug.loc[N_FOLDS] = [
    "Mean / Std",
    f"{np.mean(xgb_aug_aucs):.5f} / {np.std(xgb_aug_aucs):.5f}",
]
display(df_xgb_aug.set_index("Fold"))


XGBoost HVRT-augmented  --  wall time: 0.7s



,AUC
Fold,
Fold 1,0.85833
Fold 2,0.89653
Fold 3,0.8
Fold 4,0.89028
Fold 5,0.88472
Mean / Std,0.86597 / 0.03546


In [30]:
# -- Head-to-head comparison: GeoXGB vs XGBoost, before and after HVRT ------
rows = [
    {"Model": "GeoXGB",  "Pipeline": "Baseline",
     "Mean AUC": f"{np.mean(baseline_aucs):.5f}",
     "Std":      f"{np.std(baseline_aucs):.5f}"},
    {"Model": "GeoXGB",  "Pipeline": "HVRT 10k augmented",
     "Mean AUC": f"{np.mean(aug_aucs):.5f}",
     "Std":      f"{np.std(aug_aucs):.5f}",
     "Delta vs baseline": f"{np.mean(aug_aucs)-np.mean(baseline_aucs):+.5f}"},
    {"Model": "XGBoost", "Pipeline": "Baseline",
     "Mean AUC": f"{np.mean(xgb_base_aucs):.5f}",
     "Std":      f"{np.std(xgb_base_aucs):.5f}"},
    {"Model": "XGBoost", "Pipeline": "HVRT 10k augmented",
     "Mean AUC": f"{np.mean(xgb_aug_aucs):.5f}",
     "Std":      f"{np.std(xgb_aug_aucs):.5f}",
     "Delta vs baseline": f"{np.mean(xgb_aug_aucs)-np.mean(xgb_base_aucs):+.5f}"},
]
df_vs = pd.DataFrame(rows).set_index(["Model", "Pipeline"])
df_vs["Delta vs baseline"] = df_vs["Delta vs baseline"].fillna("â€”")
print("Head-to-head: GeoXGB vs XGBoost  (5-fold CV, 20 % holdout)")
print()
display(df_vs)

# GeoXGB vs XGBoost at baseline
geo_vs_xgb = np.mean(baseline_aucs) - np.mean(xgb_base_aucs)
print(f"\nGeoXGB vs XGBoost at baseline   : {geo_vs_xgb:+.5f}"
      f"  ({'GeoXGB leads' if geo_vs_xgb > 0.001 else 'XGBoost leads' if geo_vs_xgb < -0.001 else 'Tied'})")

Head-to-head: GeoXGB vs XGBoost  (5-fold CV, 20 % holdout)



Mean AUC      Std Delta vs baseline
Model   Pipeline                                              
GeoXGB  Baseline            0.86639  0.03314               â€”
        HVRT 10k augmented  0.80722  0.04356          -0.05917
XGBoost Baseline            0.86111  0.03391               â€”
        HVRT 10k augmented  0.86597  0.03546          +0.00486


GeoXGB vs XGBoost at baseline   : +0.00528  (GeoXGB leads)


---
## Stage 6 â€” 50 k Augmentation: GeoXGB and XGBoost

Repeat the augmentation experiment with `N_AUG = 50_000` to test whether
a larger synthetic pool closes the performance gap.  All four conditions
(GeoXGB/XGBoost Ã— baseline/augmented) run in parallel.

In [31]:
N_AUG_50K = 50_000

def _geo_aug_50k_worker(fold_idx, X, y, splits, geo_params, n_aug, seed):
    return _hvrt_aug_worker(fold_idx, X, y, splits, geo_params, n_aug, seed)

def _xgb_aug_50k_worker(fold_idx, X, y, splits, xgb_params, n_aug, seed):
    return _xgb_aug_worker(fold_idx, X, y, splits, xgb_params, n_aug, seed)

# 10 jobs total: GeoXGB 50k (5 folds) + XGBoost 50k (5 folds)
aug50_jobs = (
    [("geo", fi) for fi in range(N_FOLDS)] +
    [("xgb", fi) for fi in range(N_FOLDS)]
)

print(f"Launching {len(aug50_jobs)} jobs (GeoXGB 50k + XGBoost 50k) across {N_JOBS} workers...")
t0 = time.perf_counter()

aug50_raw = Parallel(n_jobs=N_JOBS, prefer="processes")(
    delayed(_geo_aug_50k_worker if tag == "geo" else _xgb_aug_50k_worker)(
        fi, X, y, splits,
        GEO_DEFAULTS if tag == "geo" else XGB_PARAMS,
        N_AUG_50K, RANDOM_STATE
    )
    for tag, fi in aug50_jobs
)

aug50_wall = time.perf_counter() - t0
print(f"Done. Wall time: {aug50_wall:.1f}s")

geo50_raw = sorted(aug50_raw[:N_FOLDS],  key=lambda r: r[0])
xgb50_raw = sorted(aug50_raw[N_FOLDS:], key=lambda r: r[0])

geo50_aucs = [r[1] for r in geo50_raw]
xgb50_aucs = [r[1] for r in xgb50_raw]

print()
print(f"GeoXGB  50k: {[round(a,5) for a in geo50_aucs]}")
print(f"XGBoost 50k: {[round(a,5) for a in xgb50_aucs]}")

Launching 10 jobs (GeoXGB 50k + XGBoost 50k) across 32 workers...
Done. Wall time: 269.1s

GeoXGB  50k: [0.76944, 0.79375, 0.60764, 0.82431, 0.78889]
XGBoost 50k: [0.88542, 0.89306, 0.80417, 0.88889, 0.84722]


In [32]:
# -- 10k vs 50k comparison table ---------------------------------------------
rows50 = [
    {"Model": "GeoXGB",  "N_AUG": "baseline",
     "Mean AUC": np.mean(baseline_aucs),  "Std": np.std(baseline_aucs),  "Delta": None},
    {"Model": "GeoXGB",  "N_AUG": "10 k",
     "Mean AUC": np.mean(aug_aucs),       "Std": np.std(aug_aucs),
     "Delta": np.mean(aug_aucs)   - np.mean(baseline_aucs)},
    {"Model": "GeoXGB",  "N_AUG": "50 k",
     "Mean AUC": np.mean(geo50_aucs),     "Std": np.std(geo50_aucs),
     "Delta": np.mean(geo50_aucs) - np.mean(baseline_aucs)},
    {"Model": "XGBoost", "N_AUG": "baseline",
     "Mean AUC": np.mean(xgb_base_aucs),  "Std": np.std(xgb_base_aucs), "Delta": None},
    {"Model": "XGBoost", "N_AUG": "10 k",
     "Mean AUC": np.mean(xgb_aug_aucs),   "Std": np.std(xgb_aug_aucs),
     "Delta": np.mean(xgb_aug_aucs)  - np.mean(xgb_base_aucs)},
    {"Model": "XGBoost", "N_AUG": "50 k",
     "Mean AUC": np.mean(xgb50_aucs),     "Std": np.std(xgb50_aucs),
     "Delta": np.mean(xgb50_aucs) - np.mean(xgb_base_aucs)},
]

df50 = pd.DataFrame(rows50).set_index(["Model", "N_AUG"])
df50["Mean AUC"] = df50["Mean AUC"].map(lambda x: f"{x:.5f}")
df50["Std"]      = df50["Std"].map(lambda x: f"{x:.5f}")
df50["Delta"]    = df50["Delta"].map(lambda x: f"{x:+.5f}" if x is not None else "â€”")

print("Augmentation scaling: baseline / 10 k / 50 k  (5-fold CV, 20 % holdout)")
print()
display(df50)

Augmentation scaling: baseline / 10 k / 50 k  (5-fold CV, 20 % holdout)



Mean AUC      Std     Delta
Model   N_AUG                               
GeoXGB  baseline  0.86639  0.03314      +nan
        10 k      0.80722  0.04356  -0.05917
        50 k      0.75681  0.07663  -0.10958
XGBoost baseline  0.86111  0.03391      +nan
        10 k      0.86597  0.03546  +0.00486
        50 k      0.86375  0.03401  +0.00264